***

# **BLS Queries**

***

In this file, we want to try and pull the total number of employment from BLS for different industries. To give some context, the API works by specifying a specific series id, which can be used to pull data from a specific table from BLS. It can also be used to select information from a table, parameters like specific counties or specific industry can be referenced. The survey we are trying to pull from is State and Employment, Hours, and Earnings; which can be found in the following link: https://www.bls.gov/help/hlpforma.htm#EW. 

***

## **Preparing Workspace**

***

In [38]:
# Packages
import pandas as pd
import numpy as np
import json
import requests
import os
from functools import reduce
from tqdm import tqdm

In [39]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')
    
    # SharePoint
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
if user in ['jchoy', 'aazawii']:
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Python Projects', 'Regional-Monitoring', 'Indicator_Gen')
    path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
    path_agol = os.path.join(path_sp, 'Process Revamp', 'Task 8. Reproduce Progress Report indicators', 'Indicator Data', 'BLS Data')
    path_main = os.path.join(path_sp, 'Data')
    
path_config0 = os.path.join(path_git, 'config')
path_config  = os.path.join(path_git, 'Python Code', 'BLS', 'config')



print(user)
print(path_git)

jchoy
C:\Users\jchoy\Documents\Python Projects\Regional-Monitoring\Indicator_Gen


<>:11: SyntaxWarning: invalid escape sequence '\R'
<>:18: SyntaxWarning: invalid escape sequence '\R'
<>:11: SyntaxWarning: invalid escape sequence '\R'
<>:18: SyntaxWarning: invalid escape sequence '\R'
C:\Users\jchoy\AppData\Local\Temp\ipykernel_9760\1373476644.py:11: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')
C:\Users\jchoy\AppData\Local\Temp\ipykernel_9760\1373476644.py:18: SyntaxWarning: invalid escape sequence '\R'
  path_sp   = os.path.join(path_users, 'Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents')


In [42]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

# Base URL for API V2
url = 'https://api.bls.gov/publicAPI/v2/timeseries/data/'

# Set API key
exec(open(os.path.join(path_config, 'api_key.txt')).read())
key = dict_api[user]
key = '?registrationkey={}'.format(key)

<string>:1539: SyntaxWarning: invalid escape sequence '\$'
<string>:1541: SyntaxWarning: invalid escape sequence '\$'


***

## **Preparing Imports**

***

In [ ]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

***

## **Importing**

***

In [ ]:
# Query BLS data based on parameters set above
# Convert total jobs to raw counts (instead of per thousand jobs)
# Reshape data 

print("Begin process pulling BLS data")
print('')

dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = list_sectors
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp


# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

def merge_dfs(df1, df2):
    return df1.merge(df2, on=['date_', 'Geography'])
df_joined = reduce(merge_dfs, dfs)
df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
                    var_name='Industry', 
                    value_name='Value')
var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

df_bls1['date_'] = df_bls1['date_'].astype(str)
df_bls2['date_'] = df_bls2['date_'].astype(str)

df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

In [ ]:
# 4 successful pulls (almost 5 pulls - failed on 2020-2024 of the 2nd series ID)
# all peer MSAs (24)
# 2000-2024
# two series IDs (2)
5*24*2

In [ ]:
# Displaying for QC
display(df_bls1, df_bls2)

In [ ]:
# Checking to see if this matches our prior results, 

# For 2014-01-01, Sac Total Private should = 647700.0, Sac Gov = 225300.0
display(df_bls1.loc[(df_bls1['Geography'] == 'Sacramento-Roseville-Folsom, CA Metro Area') & (df_bls1['date_'] == '2014-01-01')])

In [ ]:
if percentages == 'Yes':

        # Estimate proportions by groupings
        df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[df_bls1['Value'] != 'All'].groupby(['Geography', 'date_'])['Value'].transform('sum')
            
        # Reshape data to wide format
        df_bls2_pct = df_bls1.pivot_table(index = ['Geography', 'date_']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_bls2_pct = df_bls2_pct.sort_values(['Geography', 'date_'], ascending = [True, False])


df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)
df_bls1_total['Variable'] = 'All'
df_bls1_total['Percentage'] = np.nan

df_bls1_total = pd.concat([df_bls1, df_bls1_total])
df_bls1_total = df_bls1_total.sort_values(['Geography', 'date_', 'Variable'], ascending = [True, False, True])



In [ ]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'National':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'National wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'National wide pct')


print('')
print("Successfully exported")

***

## **Jobs_2**

***

In [ ]:
# Subset 

# Go up to the first Finance, Information, Real Estate. 8 vars

# Query in sections

In [7]:
# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

Jobs_2
BLS
SMU
National
MSA
Percentages: Yes
Number of variables: 17
2000
2024


In [8]:
# Compiling all of the necessary steps into one chunk to test.

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
list_sectors = list(df_industries['industry_code'].values)
data_type    = df_datatypes['data_type_code'].values[0]

# View
print(list_sectors)
print(data_type   )


if geography == 'MSA':
    
    # Reading in MSA inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
    
    # State codes
    df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
    df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
    df_area = df_area.drop('MSA_ID', axis = 1)
    df_area = df_area.dropna()
    display(df_area.head())

if geography == 'National':

    # Reading in National Inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
    display(df_area.head())
    

['00000000', '05000000', '10000000', '20000000', '30000000', '41000000', '42000000', '43000000', '50000000', '55000000', '60000000', '65610000', '65620000', '70000000', '80000000', '90910000', '90931611']
01


,national,national_label
0,No area code needed,National


In [9]:
print("Begin process pulling BLS data")
print('')

first = list_sectors[0:7]
dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = first
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

# We will merge this with the rest of the sectors, doing only the first half now as it requires too much computation

dfs_beta = dfs
# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date_', 'Geography'])
# df_joined = reduce(merge_dfs, dfs)
# df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
#                     var_name='Industry', 
#                     value_name='Value')
# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
# df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
# df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
# df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
# df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

# df_bls1['date_'] = df_bls1['date_'].astype(str)
# df_bls2['date_'] = df_bls2['date_'].astype(str)

# df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
# df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

Begin process pulling BLS data


Creating python dictionary of industry IDs



100%|██████████| 7/7 [00:00<?, ?it/s]


Pulling data for each industry ID by decade


{'SMU0000000001': 'National'}

Pulling data from 2000 to 2009


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 996.27it/s]

...

100%|██████████| 1/1 [00:00<?, ?it/s]


Pulling data from 2020 to 2024


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<?, ?it/s]


Reshaping pulled data


Finished ^_^..V..


In [58]:
print("Begin process pulling BLS data")
print('')

second = list_sectors[7:15]
dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = second
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

# We will merge this with the rest of the sectors, doing only the first half now as it requires too much computation

dfs_nu = dfs
# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date_', 'Geography'])
# df_joined = reduce(merge_dfs, dfs)
# df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
#                     var_name='Industry', 
#                     value_name='Value')
# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
# df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
# df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
# df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
# df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

# df_bls1['date_'] = df_bls1['date_'].astype(str)
# df_bls2['date_'] = df_bls2['date_'].astype(str)

# df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
# df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

Begin process pulling BLS data


Creating python dictionary of industry IDs



100%|██████████| 8/8 [00:00<00:00, 7979.65it/s]


Pulling data for each industry ID by decade


{'SMU4300000001': 'National'}

Pulling data from 2000 to 2009


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 185.71it/s]

...

REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 171.16it/s]


Pulling data from 2020 to 2024


Request could not be serviced, as the daily threshold for total number of requests allocated to the user with registration key  has been reached.

{'SMU5500000001': 'National'}

Pulling data from 2000 to 2009
Request could not be serviced, as the daily threshold for total number of requests allocated to the user with registration key  has been reached.


ValueError: No objects to concatenate

In [ ]:
# rerun this again tomo and next day

sheet_names = ['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6', 'Sheet7', 'Sheet8']

dataframe_path = os.path.join(path_config, 'nat 2.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_nu, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"\nDataFrames have been exported to {dataframe_path}")

In [12]:
# rerun this again tomo and next day

sheet_names = ['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6', 'Sheet7', 'Sheet8']

dataframe_path = os.path.join(path_config, 'multiple_dataframes_2.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_beta, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"\nDataFrames have been exported to {dataframe_path}")


DataFrames have been exported to C:\Users\jchoy\Documents\Python Projects\Regional-Monitoring\Indicator_Gen\Python Code\BLS\config\multiple_dataframes.xlsx


In [ ]:
# Now re-run all prior code and then merge between df_bls1_total and df_jobs_2_beta

# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

In [ ]:
# Compiling all of the necessary steps into one chunk to test.

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
list_sectors = list(df_industries['industry_code'].values)
data_type    = df_datatypes['data_type_code'].values[0]

# View
print(list_sectors)
print(data_type   )


if geography == 'MSA':
    
    # Reading in MSA inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
    
    # State codes
    df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
    df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
    df_area = df_area.drop('MSA_ID', axis = 1)
    df_area = df_area.dropna()
    display(df_area.head())

if geography == 'National':

    # Reading in National Inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
    display(df_area.head())
    

In [ ]:
print("Begin process pulling BLS data")
print('')

dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = list_sectors
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

dfs_gamma = dfs
dfs = dfs_beta + dfs

sheet_names = ['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6']

dataframe_path = os.path.join(path_config, 'multiple_dataframes_2.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_gamma, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"\nDataFrames have been exported to {dataframe_path}")

# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date_', 'Geography'])
# df_joined = reduce(merge_dfs, dfs)
# df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
#                     var_name='Industry', 
#                     value_name='Value')
# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
# df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
# df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
# df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
# df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

# df_bls1['date_'] = df_bls1['date_'].astype(str)
# df_bls2['date_'] = df_bls2['date_'].astype(str)

# df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
# df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

In [ ]:
sheet_names = ['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6']

dataframe_path = os.path.join(path_config, 'multiple_dataframes_2.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_gamma, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

In [69]:
# Now concatenate dfs into df_beta + df_gamma + df_lambda

# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

Jobs_2
BLS
SMU
MSA
MSA
Percentages: Yes
Number of variables: 17
2000
2024


In [71]:
# Compiling all of the necessary steps into one chunk to test.

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
list_sectors = list(df_industries['industry_code'].values)
data_type    = df_datatypes['data_type_code'].values[0]

# View
print(list_sectors)
print(data_type   )


if geography == 'MSA':
    
    # Reading in MSA inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
    
    # State codes
    df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
    df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
    df_area = df_area.drop('MSA_ID', axis = 1)
    df_area = df_area.dropna()
    display(df_area.head())

if geography == 'National':

    # Reading in National Inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
    display(df_area.head())
    

['00000000', '05000000', '10000000', '20000000', '30000000', '41000000', '42000000', '43000000', '50000000', '55000000', '60000000', '65610000', '65620000', '70000000', '80000000', '90910000', '90931611']
01


,national,national_label
0,No area code needed,National


In [ ]:
print("Begin process pulling BLS data")
print('')

dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = list_sectors
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

dfs_lambda = dfs

sheet_names = ['Sheet1', 'Sheet2', 'Sheet3']

dataframe_path = os.path.join(path_config, 'multiple_dataframes_3.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_lambda, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

print(f"\nDataFrames have been exported to {dataframe_path}")

print('')
print("Finished ^_^..V..")

In [50]:
# merge all dataframes togetehr from the multiple frames 1, 2, 3 

for sheet in sheet_names:
    df = pd.read_excel(dataframe_path, sheet_name=sheet)
    mult_files.append(df)

In [64]:
# Merge all together | Only used 7 series, but we need 17 for the mapping. So rerun the inputs code again

def merge_dfs(df1, df2):
    return df1.merge(df2, on=['date_', 'Geography'])
df_joined = reduce(merge_dfs, mult_files)
df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
                    var_name='Industry', 
                    value_name='Value')
var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

df_bls1['date_'] = df_bls1['date_'].astype(str)
df_bls2['date_'] = df_bls2['date_'].astype(str)

df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])

In [65]:
df_bls1

,date_,Geography,Variable,Value
94024,2024-05-01,"Austin-Round Rock-Georgetown, TX Metro Area",Construction,0.0
94025,2024-05-01,"Austin-Round Rock-Georgetown, TX Metro Area",Education,0.0
94026,2024-05-01,"Austin-Round Rock-Georgetown, TX Metro Area",Federal Government,15000.0
94027,2024-05-01,"Austin-Round Rock-Georgetown, TX Metro Area","Finance, Information, Real Estate",134200.0
94028,2024-05-01,"Austin-Round Rock-Georgetown, TX Metro Area",Health,133800.0
...,...,...,...,...
317,2000-01-01,"Yuba City, CA Metro Area",Professional and Business Services,2600.0
318,2000-01-01,"Yuba City, CA Metro Area",Retail,5300.0
319,2000-01-01,"Yuba City, CA Metro Area",Total Nonfarm,35200.0
320,2000-01-01,"Yuba City, CA Metro Area",Total Private,24600.0


In [66]:
if percentages == 'Yes':

        # Estimate proportions by groupings
        df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[df_bls1['Value'] != 'All'].groupby(['Geography', 'date_'])['Value'].transform('sum')
            
        # Reshape data to wide format
        df_bls2_pct = df_bls1.pivot_table(index = ['Geography', 'date_']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_bls2_pct = df_bls2_pct.sort_values(['Geography', 'date_'], ascending = [True, False])


df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)
df_bls1_total['Variable'] = 'All'
df_bls1_total['Percentage'] = np.nan

df_bls1_total = pd.concat([df_bls1, df_bls1_total])
df_bls1_total = df_bls1_total.sort_values(['Geography', 'date_', 'Variable'], ascending = [True, False, True])



C:\Users\jchoy\AppData\Local\Temp\ipykernel_31428\3214454971.py:13: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)


In [67]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

Jobs_2 MSA BLS SMU.xlsx
Jobs_2_MSA_BLS_SMU.csv


In [68]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'National':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'National wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'National wide pct')


print('')
print("Successfully exported")

Excel files exported here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Vibrant and Inclusive Places\Economy\\Jobs\\Jobs_1 Total
CSV files exported here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Process Revamp\Task 8. Reproduce Progress Report indicators\Indicator Data\BLS Data\Jobs_2

Successfully exported


***

## **Jobs 3**

***

In [37]:
# Total Private, Goods Producing Service Providing

# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

Jobs_3
BLS
SMU
MSA
MSA
Percentages: Yes
Number of variables: 3
2000
2024


In [38]:
# Compiling all of the necessary steps into one chunk to test.

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
list_sectors = list(df_industries['industry_code'].values)
data_type    = df_datatypes['data_type_code'].values[0]

# View
print(list_sectors)
print(data_type   )


if geography == 'MSA':
    
    # Reading in MSA inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
    
    # State codes
    df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
    df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
    df_area = df_area.drop('MSA_ID', axis = 1)
    df_area = df_area.dropna()
    display(df_area.head())

if geography == 'National':

    # Reading in National Inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
    display(df_area.head())
    

['05000000', '06000000', '07000000', '90000000']
01


,msa,msa_label,area_code,State FIPS
0,12420,"Austin-Round Rock-Georgetown, TX Metro Area",MT4812420000000,48
1,16740,"Charlotte-Concord-Gastonia, NC-SC Metro Area",MT3716740000000,37
2,17140,"Cincinnati, OH-KY-IN Metro Area",MT3917140000000,39
3,17460,"Cleveland-Elyria, OH Metro Area",MT3917460000000,39
4,18140,"Columbus, OH Metro Area",MT3918140000000,39


In [39]:
print("Begin process pulling BLS data")
print('')

dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = list_sectors
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp


# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

def merge_dfs(df1, df2):
    return df1.merge(df2, on=['date_', 'Geography'])
df_joined = reduce(merge_dfs, dfs)
df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
                    var_name='Industry', 
                    value_name='Value')
var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

df_bls1['date_'] = df_bls1['date_'].astype(str)
df_bls2['date_'] = df_bls2['date_'].astype(str)

df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

Begin process pulling BLS data


Creating python dictionary of industry IDs



100%|██████████| 4/4 [00:00<00:00, 585.22it/s]


Pulling data for each industry ID by decade


{'SMU48124200500000001': 'Austin-Round Rock-Georgetown, TX Metro Area', 'SMU37167400500000001': 'Charlotte-Concord-Gastonia, NC-SC Metro Area', 'SMU39171400500000001': 'Cincinnati, OH-KY-IN Metro Area', 'SMU39174600500000001': 'Cleveland-Elyria, OH Metro Area', 'SMU39181400500000001': 'Columbus, OH Metro Area', 'SMU26198200500000001': 'Detroit-Warren-Dearborn, MI Metro Area', 'SMU18269000500000001': 'Indianapolis-Carmel-Anderson, IN Metro Area', 'SMU29281400500000001': 'Kansas City, MO-KS Metro Area', 'SMU12331000500000001': 'Miami-Fort Lauderdale-Pompano Beach, FL Metro Area', 'SMU12367400500000001': 'Orlando-Kissimmee-Sanford, FL Metro Area', 'SMU04380600500000001': 'Phoenix-Mesa-Chandler, AZ Metro Area', 'SMU42383000500000001': 'Pittsburgh, PA Metro Area', 'SMU41389000500000001': 'Portland-Vancouver-Hillsboro, OR-WA Metro Area', 'SMU06401400500000001': 'Riverside-San Bernardino-Ontario, CA Metro Area', 'SMU06409000500000001': 'Sacrament

REQUEST_SUCCEEDED


100%|██████████| 23/23 [00:00<00:00, 482.61it/s]

...

100%|██████████| 23/23 [00:00<00:00, 1124.82it/s]


Pulling data from 2020 to 2024


REQUEST_SUCCEEDED


100%|██████████| 23/23 [00:00<00:00, 1182.80it/s]


Reshaping pulled data


Finished ^_^..V..


In [40]:
# Checking to see if this matches our prior results, 

# For 2014-01-01, Sac Total Private should = 647700.0, Sac Gov = 225300.0
display(df_bls1.loc[(df_bls1['Geography'] == 'Sacramento-Roseville-Folsom, CA Metro Area') & (df_bls1['date_'] == '2014-01-01')])

,date_,Geography,Variable,Value
15512,2014-01-01,"Sacramento-Roseville-Folsom, CA Metro Area",Goods Producing,76800.0
15513,2014-01-01,"Sacramento-Roseville-Folsom, CA Metro Area",Government,225300.0
15514,2014-01-01,"Sacramento-Roseville-Folsom, CA Metro Area",Service-Providing,796200.0
15515,2014-01-01,"Sacramento-Roseville-Folsom, CA Metro Area",Total Private,647700.0


In [41]:
if percentages == 'Yes':

        # Estimate proportions by groupings
        df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[df_bls1['Value'] != 'All'].groupby(['Geography', 'date_'])['Value'].transform('sum')
            
        # Reshape data to wide format
        df_bls2_pct = df_bls1.pivot_table(index = ['Geography', 'date_']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_bls2_pct = df_bls2_pct.sort_values(['Geography', 'date_'], ascending = [True, False])


df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)
df_bls1_total['Variable'] = 'All'
df_bls1_total['Percentage'] = np.nan

df_bls1_total = pd.concat([df_bls1, df_bls1_total])
df_bls1_total = df_bls1_total.sort_values(['Geography', 'date_', 'Variable'], ascending = [True, False, True])



C:\Users\jchoy\AppData\Local\Temp\ipykernel_21180\3214454971.py:13: FutureWarning: The provided callable <built-in function sum> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)


In [42]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

Jobs_3 MSA BLS SMU.xlsx
Jobs_3_MSA_BLS_SMU.csv


In [43]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'National':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'National wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'National wide pct')


print('')
print("Successfully exported")

Excel files exported here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Data\Vibrant and Inclusive Places\Economy\\Jobs\\Jobs_1 Total
CSV files exported here: C:\Users\jchoy\Sacramento Area Council of Governments\Regional Monitoring and Reporting - Documents\Process Revamp\Task 8. Reproduce Progress Report indicators\Indicator Data\BLS Data\Jobs_3

Successfully exported


***

## **National Jobs_2 and Jobs_3**

***

In [26]:
# Rerun this code but change inputs ezpz

# Import objects
df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

Jobs_2
BLS
CEU
National
MSA
Percentages: Yes
Number of variables: 17
2000
2024


In [27]:
# Compiling all of the necessary steps into one chunk to test.

# Industries and data types
df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                              , sheet_name = 'industry_codes'
                              , dtype = {'industry_code': object})
df_industries = df_industries[df_industries['Include'] == 'Yes']
df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                             , sheet_name = 'datatype_codes'
                              , dtype = {'data_type_code': object})
df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# Inputs
list_sectors = list(df_industries['industry_code'].values)
data_type    = df_datatypes['data_type_code'].values[0]

# View
print(list_sectors)
print(data_type   )


if geography == 'MSA':
    
    # Reading in MSA inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
    
    # State codes
    df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
    df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
    df_area = df_area.drop('MSA_ID', axis = 1)
    df_area = df_area.dropna()
    display(df_area.head())

if geography == 'National':

    # Reading in National Inputs
    df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
    display(df_area.head())
    

['00000000', '05000000', '10000000', '20000000', '30000000', '41000000', '42000000', '43000000', '50000000', '55000000', '60000000', '65610000', '65620000', '70000000', '80000000', '90910000', '90931611']
01


,national,national_label
0,No area code needed,National


In [35]:
list_sectors[7:14]

['43000000',
 '50000000',
 '55000000',
 '60000000',
 '65610000',
 '65620000',
 '70000000']

In [33]:
list_sectors

['00000000',
 '05000000',
 '10000000',
 '20000000',
 '30000000',
 '41000000',
 '42000000',
 '43000000',
 '50000000',
 '55000000',
 '60000000',
 '65610000',
 '65620000',
 '70000000',
 '80000000',
 '90910000',
 '90931611']

In [36]:
print("Begin process pulling BLS data")
print('')

batch_3 = list_sectors[14:]
dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = batch
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

dfs_nat_1 = dfs

# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date_', 'Geography'])
# df_joined = reduce(merge_dfs, dfs)
# df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
#                     var_name='Industry', 
#                     value_name='Value')
# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
# df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
# df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
# df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
# df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

# df_bls1['date_'] = df_bls1['date_'].astype(str)
# df_bls2['date_'] = df_bls2['date_'].astype(str)

# df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
# df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

Begin process pulling BLS data


Creating python dictionary of industry IDs



100%|██████████| 7/7 [00:00<00:00, 815.08it/s]


Pulling data for each industry ID by decade


{'CEU4300000001': 'National'}

Pulling data from 2000 to 2009


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 28.27it/s]

...

100%|██████████| 1/1 [00:00<00:00, 211.55it/s]


Pulling data from 2020 to 2024


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 365.33it/s]


Reshaping pulled data


Finished ^_^..V..


In [37]:
dfs_nat_2 = dfs_nat_1

In [15]:
list_sectors[14:]

['80000000', '90910000', '90931611']

In [18]:
print("Begin process pulling BLS data")
print('')

batch_3 = list_sectors[14:]
dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = batch_3
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

ind_list = df_industries['industry_name'].values.tolist()

print('')
print("Reshaping pulled data")
print('')

for ii in range(len(dfs)):
    df_temp = dfs[ii]
    for col in df_temp.columns:
        df_temp[col] = df_temp[col].apply(lambda x: x*1000)
    df_temp = df_temp.reset_index(names = 'date_')
    df_temp = pd.melt(df_temp
               , id_vars = 'date_'
               , var_name = 'Geography'
               , value_name = ind_list[ii])
    dfs[ii] = df_temp

dfs_nat_3 = dfs

# Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
# Each "Jobs" column should be named by sector code description
# Roll up total jobs in each specific industry codes to the mapped sectors
# Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
# Clean date field

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date_', 'Geography'])
# df_joined = reduce(merge_dfs, dfs)
# df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
#                     var_name='Industry', 
#                     value_name='Value')
# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
# df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
# df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
# df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
# df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

# df_bls1['date_'] = df_bls1['date_'].astype(str)
# df_bls2['date_'] = df_bls2['date_'].astype(str)

# df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
# df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])


print('')
print("Finished ^_^..V..")

Begin process pulling BLS data


Creating python dictionary of industry IDs



  0%|          | 0/3 [00:00<?, ?it/s]

100%|██████████| 3/3 [00:00<00:00, 436.21it/s]


Pulling data for each industry ID by decade


{'SMU8000000001': 'National'}

Pulling data from 2000 to 2009


REQUEST_SUCCEEDED


...

100%|██████████| 1/1 [00:00<?, ?it/s]


Pulling data from 2020 to 2024


REQUEST_SUCCEEDED


100%|██████████| 1/1 [00:00<00:00, 328.91it/s]


Reshaping pulled data


Finished ^_^..V..


In [14]:
sheet_names = ['Sheet1', 'Sheet2', 'Sheet3', 'Sheet4', 'Sheet5', 'Sheet6', 'Sheet7']

dataframe_path = os.path.join(path_config, 'nat_2.xlsx')

# Using ExcelWriter to write DataFrames to different sheets
with pd.ExcelWriter(dataframe_path, engine='openpyxl') as writer:
    for df, sheet_name in zip(dfs_nat_2, sheet_names):
        df.to_excel(writer, index=False, sheet_name=sheet_name)

In [ ]:
def merge_dfs(df1, df2):
    return df1.merge(df2, on=['date_', 'Geography'])
df_joined = reduce(merge_dfs, dfs)
df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
                    var_name='Industry', 
                    value_name='Value')
var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

df_bls1['date_'] = df_bls1['date_'].astype(str)
df_bls2['date_'] = df_bls2['date_'].astype(str)

df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])

In [ ]:
if percentages == 'Yes':

        # Estimate proportions by groupings
        df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[df_bls1['Value'] != 'All'].groupby(['Geography', 'date_'])['Value'].transform('sum')
            
        # Reshape data to wide format
        df_bls2_pct = df_bls1.pivot_table(index = ['Geography', 'date_']
                                           , columns = 'Variable'
                                           , values = 'Percentage').reset_index()
        df_bls2_pct = df_bls2_pct.sort_values(['Geography', 'date_'], ascending = [True, False])


df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)
df_bls1_total['Variable'] = 'All'
df_bls1_total['Percentage'] = np.nan

df_bls1_total = pd.concat([df_bls1, df_bls1_total])
df_bls1_total = df_bls1_total.sort_values(['Geography', 'date_', 'Variable'], ascending = [True, False, True])



In [ ]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

In [ ]:
# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'National':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'National wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'National wide pct')


print('')
print("Successfully exported")

***

## **LAU Pulls**

***

In [ ]:
# Preparing inputs again. 

df_params = pd.read_excel(os.path.join(path_config, 'BLS Configuration File.xlsx'), sheet_name = 'Inputs')

# Set parameters for querying ACS data
indicator_name     = df_params[df_params['Type'] == 'indicator_name'    ]['Input'].values[0]
estimate           = df_params[df_params['Type'] == 'estimate'          ]['Input'].values[0]
survey             = df_params[df_params['Type'] == 'sample'            ]['Input'].values[0]
geography          = df_params[df_params['Type'] == 'geography'         ]['Input'].values[0]
import_tab         = df_params[df_params['Type'] == 'import_tab'        ]['Input'].values[0]
percentages        = df_params[df_params['Type'] == 'percentages'       ]['Input'].values[0]
num_vars           = df_params[df_params['Type'] == 'num_vars'          ]['Input'].values[0]
year_start         = df_params[df_params['Type'] == 'year_start'        ]['Input'].values[0]
year_end           = df_params[df_params['Type'] == 'year_end'          ]['Input'].values[0]
report_theme       = df_params[df_params['Type'] == 'report_theme'      ]['Input'].values[0]
sp_folder_out      = df_params[df_params['Type'] == 'sp_folder'         ]['Input'].values[0]


# view
print(indicator_name)
print(estimate)
print(survey)
print(geography)
print(import_tab)
print("Percentages: " + percentages)
print("Number of variables: " + str(num_vars))
print(year_start)
print(year_end)

In [ ]:
# Compiling all of the necessary steps into one chunk to test.

if survey in ('SMU', 'CEU'):

    # Industries and data types
    df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                                , sheet_name = 'industry_codes'
                                , dtype = {'industry_code': object})
    df_industries = df_industries[df_industries['Include'] == 'Yes']
    df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
    df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                                , sheet_name = 'datatype_codes'
                                , dtype = {'data_type_code': object})
    df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
    df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

    # Inputs
    list_sectors = list(df_industries['industry_code'].values)
    data_type    = df_datatypes['data_type_code'].values[0]

    print(list_sectors)
    print(data_type   )

    if geography == 'MSA':
        
        # Reading in MSA inputs
        df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})

        # State codes
        df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object, 'MSA_ID': object})
        df_area = df_area.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
        df_area = df_area.drop('MSA_ID', axis = 1)
        df_area = df_area.dropna()
        display(df_area.head())

    if geography == 'National':

        # Reading in National Inputs
        df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'National', dtype = {'national': object})
        display(df_area.head())

if survey == 'LAU':

    # Industries and data types
    df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                                , sheet_name = 'industry_codes'
                                , dtype = {'industry_code': object})
    df_industries = df_industries[df_industries['Include'] == 'Yes']
    df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)]
    df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
                                , sheet_name = 'datatype_codes'
                                , dtype = {'data_type_code': object})
    df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
    df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]
    data_type    = df_datatypes['data_type_code'].values[0]

    if geography == 'MSA':
        # Reading in MSA inputs
        df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'MSA', dtype = {'msa': object})
        list_sectors = list(df_area['area_code'])

    if geography == 'County':
        df_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'County', dtype = {'msa': object})
        list_sectors = list(df_area['county_code'])

    print(list_sectors)
    print(data_type   )


In [ ]:
# Now time to actually pull. 

# To do this correctly, df_area for MSA should have msa_label be the values, whilst area_code is the keys

print("Begin process pulling BLS data!! (づ ◕‿◕ )づ")
print('')

dfs = full_bls(key = dict_api[user]
               , df = df_area
               , geography = geography
               , sector_list = list_sectors
               , dates = (year_start, year_end)
               , survey = survey
               , data_type = data_type)

if survey in ('SMU', 'CEU'):
    ind_list = df_industries['industry_name'].values.tolist()

    print('')
    print("Reshaping pulled data")
    print('')

    for ii in range(len(dfs)):
        df_temp = dfs[ii]
        for col in df_temp.columns:
            df_temp[col] = df_temp[col].apply(lambda x: x*1000)
        df_temp = df_temp.reset_index(names = 'date_')
        df_temp = pd.melt(df_temp
                , id_vars = 'date_'
                , var_name = 'Geography'
                , value_name = ind_list[ii])
        dfs[ii] = df_temp


    # Merge every single dataframe we will have together (should maybe includes a `how = 'left'`?)
    # Each "Jobs" column should be named by sector code description
    # Roll up total jobs in each specific industry codes to the mapped sectors
    # Organize two shapes of data frames - "Long" (melted) and "Wide" (dcasted)
    # Clean date field

    def merge_dfs(df1, df2):
        return df1.merge(df2, on=['date_', 'Geography'])
    df_joined = reduce(merge_dfs, dfs)
    df_bls1 = df_joined.melt(id_vars=['date_', 'Geography'], 
                        var_name='Industry', 
                        value_name='Value')
    var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))
    df_bls1['Variable'] = df_bls1['Industry'].map(var_map)
    df_bls1 = df_bls1[['date_', 'Geography', 'Industry', 'Variable', 'Value']]
    df_bls1 = df_bls1.groupby(['date_', 'Geography', 'Variable'])['Value'].sum().reset_index()
    df_bls2 = (df_bls1.pivot_table(index=['Geography', 'date_'], columns='Variable', values='Value')).reset_index()

    df_bls1['date_'] = df_bls1['date_'].astype(str)
    df_bls2['date_'] = df_bls2['date_'].astype(str)

    df_bls1 = df_bls1.sort_values(['Geography', 'date_'], ascending = [True, False])
    df_bls2 = df_bls2.sort_values(['Geography', 'date_'], ascending = [True, False])

if survey == 'LAU':
    # Assigning
    df_ump = dfs[0]

    # Making it so that date is now a column and not an index, dropping the new index, and then sorting
    df_ump.reset_index(inplace = True)
    df_ump = df_ump.rename(columns={"index": "date"})
    df_ump = df_ump.sort_values(by = 'date')
    
print('')
print("Finished ^_^..V..")

In [ ]:
if percentages == 'Yes':
        if survey == 'LAU':
                dates = df_ump.date
                df_ump = df_ump.drop(columns='date')
                if geography == 'County':
                        df_ump = df_ump.apply(pd.to_numeric, errors='coerce')
                df_ump = df_ump.applymap(lambda x: f"{float(x)}%" if not pd.isna(x) else "NaN%")
                df_ump = pd.concat([pd.Series(dates, name='date'), df_ump], axis=1)

        if survey in ('SMU', 'CEU'):
                df_bls1['Percentage'] = 100*df_bls1['Value'] / df_bls1[df_bls1['Value'] != 'All'].groupby(['Geography', 'date_'])['Value'].transform('sum')
                
                # Reshape data to wide format
                df_bls2_pct = df_bls1.pivot_table(index = ['Geography', 'date_']
                                                , columns = 'Variable'
                                                , values = 'Percentage').reset_index()
                df_bls2_pct = df_bls2_pct.sort_values(['Geography', 'date_'], ascending = [True, False])


                df_bls1_total = df_bls1.groupby(['date_', 'Geography'], as_index = False)['Value'].agg(sum)
                df_bls1_total['Variable'] = 'All'
                df_bls1_total['Percentage'] = np.nan

                df_bls1_total = pd.concat([df_bls1, df_bls1_total])
                df_bls1_total = df_bls1_total.sort_values(['Geography', 'date_', 'Variable'], ascending = [True, False, True])



In [ ]:
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print(path_out_xlsx)
print(path_out_csv)

In [ ]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

In [ ]:
# Set file path for exporting
# path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_ump.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

# Export to excel

if geography == 'MSA':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        if survey == 'LAU':
            if percentages == 'Yes':
                df_ump      .to_excel(writer, index = False, sheet_name = 'MSA PCT'  )
            else:
                df_ump      .to_excel(writer, index = False, sheet_name = 'MSA'  )
        if survey in ('SMU', 'CEU'):
            df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
            df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
            if percentages == 'Yes':
                df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'County':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        if survey == 'LAU':
            if percentages == 'Yes':
                df_ump      .to_excel(writer, index = False, sheet_name = 'County PCT'  )
            else:
                df_ump      .to_excel(writer, index = False, sheet_name = 'County'  )
        if survey in ('SMU', 'CEU'):
            # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
            df_bls1_total.to_excel(writer, index = False, sheet_name = 'MSA'       )
            df_bls2      .to_excel(writer, index = False, sheet_name = 'MSA wide'  )
            if percentages == 'Yes':
                df_bls2_pct.to_excel(writer, index = False, sheet_name = 'MSA wide pct')

if geography == 'National':
    with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
        df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )
        df_bls2      .to_excel(writer, index = False, sheet_name = 'National wide'  )
        if percentages == 'Yes':
            df_bls2_pct.to_excel(writer, index = False, sheet_name = 'National wide pct')
print('')
print("Successfully exported")

***

## **Code Graveyard**


***

In [ ]:
# https://api.bls.gov/publicAPI/v2/timeseries/data/SMU06409009093262201

In [ ]:
# # Series stored as a dictionary (unemployment rate by ethnicity)
# series_dict = {
#     'LNS14000003': 'White',
#     'LNS14000006': 'Black',
#     'LNS14000009': 'Hispanic'}

# # series_dict = {'SMU48124200500000001': 'Total Jobs'}

# # Start year and end year
# dates = ('2008', '2017')

In [ ]:
# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid": list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear": dates[1]})

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']

In [ ]:
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()

In [ ]:
# # Series stored as a dictionary
# series_dict = {
#     # 'LNS12000000': 'Agricultural Total Employment'
#     'SMU48124202023800001': 'Agricultural Total Employment1'
#     , 'SMU06409002023800001': 'Agricultural Total Employment2'
# }

# # Start year and end year
# dates = ('2022', '2024')

# # Specify json as content type to return
# headers = {'Content-type': 'application/json'}

# # Submit the list of series as data
# data = json.dumps({
#     "seriesid" : list(series_dict.keys()),
#     "startyear": dates[0],
#     "endyear"  : dates[1]
# })

# # Post request for the data
# p = requests.post(
#     '{}{}'.format(url, key),
#     headers=headers,
#     data=data).json()['Results']['series']
# # Date index from first series
# date_list = [f"{i['year']}-{i['period'][1:]}-01" for i in p[0]['data']]

# # Empty dataframe to fill with values
# df = pd.DataFrame()

# # Build a pandas series from the API results, p
# for s in p:
#     df[series_dict[s['seriesID']]] = pd.Series(
#         index = pd.to_datetime(date_list),
#         data = [i['value'] for i in s['data']]
#         ).astype(float).iloc[::-1]

# # Show last 5 results
# df.tail()


In [ ]:
# Reading in MSA inputs
# df_params = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'BLS_MSA')
# indicator_name = df_params['indicator_name'].values[0]
# year_start     = int(df_params['year_start'    ].values[0])
# year_end       = int(df_params['year_end'      ].values[0])


# df_peer_msa = df_params[['msa', 'msa_label']]

# df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object})
# df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
# df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)

# print(indicator_name)
# print(year_start    )
# print(year_end      )
# df_peer_msa.head()

In [ ]:
# df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                               , sheet_name = 'industry_codes'
#                               , dtype = {'industry_code': object})
# df_industries = df_industries[df_industries['Include'] == 'Yes']
# df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)] # Jobs_2
# df_industries.head()

In [ ]:
# df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                              , sheet_name = 'datatype_codes'
#                               , dtype = {'data_type_code': object})
# df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
# df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]
# df_datatypes.head()

In [ ]:
# # Subset
# df_peer_msa_in = df_peer_msa[df_peer_msa['msa_label'].str.contains('Sac|Austin')]
# df_peer_msa_in

In [ ]:
# Testing Agg = Total Nonfarm - Total Priv
# sectors = ['00000000', '05000000', '08000000']

# Inputs
# sectors    = list(df_industries['industry_code'].values)
# year_start = int(df_params['year_start'].values[0])
# year_end   = int(df_params['year_end'  ].values[0])
# data_type  = df_datatypes['data_type_code'].values[0]

# print(sectors   )
# print(year_start)
# print(year_end  )
# print(data_type )

# dfs = full_bls(key = dict_api[user]
#                , sector_list = sectors
#                , df = df_peer_msa_in
#                , dates = (year_start, year_end)
#                , pre = "SMU" # defines the survey
#                , data_type = data_type)

In [ ]:
# dfs_test[0] = dfs_test[0].rename(columns = {'Total Jobs':'05000000'})
# dfs_test[1] = dfs_test[1].rename(columns = {'Total Jobs':'90000000'})



# df_final = dfs_test[0].merge(dfs_test[1], on = ['date', 'MSA'])


In [ ]:
# from functools import reduce

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date', 'MSA'])

# # Use reduce to merge the entire list of DataFrames
# df_final = reduce(merge_dfs, dfs_test)

In [ ]:
# DOING JOBS 2

# Reading in MSA inputs
# df_params = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx"), sheet_name = 'BLS_MSA')
# indicator_name = df_params['indicator_name'].values[0]
# year_start     = int(df_params['year_start'    ].values[0])
# year_end       = int(df_params['year_end'      ].values[0])


# df_peer_msa = df_params[['msa', 'msa_label']]

# df_states = pd.read_excel(os.path.join(path_config0, "Area Codes.xlsx"), sheet_name = 'MSAcodes', dtype = {'State FIPS': object})
# df_peer_msa = df_peer_msa.merge(df_states[['MSA_ID', 'State FIPS']], left_on = 'msa', right_on = 'MSA_ID', how = 'left')
# df_peer_msa = df_peer_msa.drop('MSA_ID', axis = 1)

# print(indicator_name)
# print(year_start    )
# print(year_end      )
# df_peer_msa.head()

# df_industries = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                               , sheet_name = 'industry_codes'
#                               , dtype = {'industry_code': object})
# df_industries = df_industries[df_industries['Include'] == 'Yes']
# df_industries = df_industries[df_industries['Indicator Name'].str.contains(indicator_name)] # Jobs_2

# df_datatypes = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                              , sheet_name = 'datatype_codes'
#                               , dtype = {'data_type_code': object})
# df_datatypes = df_datatypes[df_datatypes['Include'] == 'Yes']
# df_datatypes = df_datatypes[df_datatypes['Indicator Name'].str.contains(indicator_name)]

# # Inputs
# sectors    = list(df_industries['industry_code'].values)
# year_start = int(df_params['year_start'].values[0])
# year_end   = int(df_params['year_end'  ].values[0])
# #data_type  = df_datatypes['data_type_code'].values[0] # I get an error here, probs cuz in the sheet it's defined as Jobs_1. 
# data_type = '01'

# print(sectors   )
# print(year_start)
# print(year_end  )
# print(data_type )

In [ ]:
# Shortening this down so we don't pull as much
# sectors = sectors[5:8]

# # have called the api 2 times today
# # dfs = full_bls(key = dict_api[user]
# #                , sector_list = sectors
# #                , df = df_peer_msa_in
# #                , dates = (year_start, year_end)
# #                , pre = "SMU" # defines the survey
# #                , data_type = data_type)


# # Goal of below chunk is to have each dataframe named by the industry it focuses on

# # Should we use Variable or Industry Name?
# ind_list = df_industries['industry_name'].values.tolist() # Change this to industry name

# # Have to subset here as well, we test more
# ind_list = ind_list[5:8]

# df_jobs_2 = dfs.copy()

# for ii in range(len(df_jobs_2)):
#     df_temp = df_jobs_2[ii]
#     for col in df_temp.columns:
#         df_temp[col] = df_temp[col].apply(lambda x: x*1000)
#     df_temp = df_temp.reset_index(names = 'date')
#     df_temp = pd.melt(df_temp
#                , id_vars = 'date'
#                , var_name = 'MSA'
#                , value_name = ind_list[ii]) # This is all i changed 
#     df_jobs_2[ii] = df_temp

# # from functools import reduce

# def merge_dfs(df1, df2):
#     return df1.merge(df2, on=['date', 'MSA'])

# # Function joins all of the Total jobs together, each of these total jobs column should be named by sector

# df_joined = reduce(merge_dfs, df_jobs_2)

# df_melted = df_joined.melt(id_vars=['date', 'MSA'], 
#                     var_name='Industry', 
#                     value_name='Value')

# # var_map is so we convert the industry names to the matching variable names we have in the excel sheet

# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))

# # mapping

# df_melted['Variable'] = df_melted['Industry'].map(var_map)

# # Industry is kept, but we won't be summing using it. 

# df_melted = df_melted[['date', 'MSA', 'Industry', 'Variable', 'Value']] #'Variable',

# # Summing matches on date, msa, and var

# df_melted = df_melted.groupby(['date', 'MSA', 'Variable'])['Value'].sum().reset_index()

# # Outputting the unmelted dataframe

# df_unmelted = (df_melted.pivot_table(index=['date', 'MSA'], columns='Variable', values='Value')).reset_index()

In [ ]:
# df_unmelted = (df_melted.pivot_table(index=['date', 'MSA'], columns='Variable', values='Value')).reset_index()

# display(df_unmelted)

In [ ]:
# df_joined = reduce(merge_dfs, df_jobs_2)

# df_melted = df_joined.melt(id_vars=['date', 'MSA'], 
#                     var_name='Industry', 
#                     value_name='Value')

# var_map = dict(zip(df_industries['industry_name'], df_industries['Variable']))

# df_melted['Variable'] = df_melted['Industry'].map(var_map)

# # df_melted = df_melted[['date', 'MSA', 'Industry', 'Variable', 'Value']] #'Variable',

# df_melted = df_melted.groupby(['date', 'MSA', 'Variable'])['Value'].sum().reset_index() # If we groupby, and don't use 4th & 5th line, it gives what we want

In [ ]:
# df_melted[(df_melted['date'] == '2023-12-01') & (df_melted['MSA'] == 'Austin-Round Rock-Georgetown, TX Metro Area')]

In [ ]:
# df_melted['Industry'].value_counts()

In [ ]:
# df_final

# Melt this dataframe, have the three columns in a single row. Have this represented by a categorical variable.

# In this example, for each date we will have three rows corresponding to the specific date

# Merge the variable mapping name onto industry_name

# So then add all of the corresponding industry_names together. For instance industry_names that fall in health will be added together by date/msa.

# Will need to use melting/decasting(this is an r function need to find the equivalent)

# See broadband_2tractacs in sharepoint https://sacog.sharepoint.com/sites/RegionalMonitoringandReporting/Shared%20Documents/Forms/AllItems.aspx?CT=1714573103380&OR=OWA%2DNT%2DMail&CID=73e9a33f%2Df757%2D7f7c%2D630e%2D591d095884dd&id=%2Fsites%2FRegionalMonitoringandReporting%2FShared%20Documents%2FData%2FVibrant%20and%20Inclusive%20Places%2FPeople%20and%20Community%2FBroadband%2FBroadband%5F2&viewid=8fba2a3f%2Dea8a%2D4663%2Da31f%2Dc1760cb6922d

In [ ]:
# Script to add area code to MSA sheet in Excel. 

# Important to note that the indexing of the columns changed in the Excel file so I adjusted the query function accordingly

# lau_area = pd.read_excel(os.path.join(path_config, "BLS Configuration File.xlsx")
#                               , sheet_name = 'lau_area'
#                              )

# # Selecting only the MSA level. We can change this if we want to do states, lmk  

# lau_area = lau_area[lau_area['area_type_code'] == 'B']

# lau_area['msa'] = lau_area['msa'].astype(str)

# df_merged = df_area.merge(lau_area[['msa', 'area_code']], on='msa', how='inner')

# df_merged = df_merged[['msa', 'msa_label', 'area_code']]

In [ ]:
# # Writing back 

# file_path = os.path.join(path_config, "BLS Configuration File.xlsx")
# with pd.ExcelWriter(file_path, engine='openpyxl', mode='a') as writer:
#     writer.book.remove(writer.book['MSA'])  # Remove the existing 'MSA' sheet
#     df_merged.to_excel(writer, sheet_name='MSA', index=False)

In [34]:
jobs_3 = pd.read_csv("Jobs_3_MSA_BLS_SM.csv")

In [35]:
jobs_3

,date_,Geography,Variable,Value
0,2024-06-01,National,Goods Producing,22085000
1,2024-06-01,National,Government,23202000
2,2024-06-01,National,Service-Providing,137307000
3,2024-06-01,National,Total Private,136190000
4,2024-05-01,National,Goods Producing,21834000
...,...,...,...,...
1171,2000-02-01,National,Total Private,108774000
1172,2000-01-01,National,Goods Producing,24080000
1173,2000-01-01,National,Government,20491000
1174,2000-01-01,National,Service-Providing,104913000


In [24]:
test = jobs_3.groupby(['date_', 'Geography', 'Variable']).sum().reset_index()

In [27]:
test = test.sort_values(by = ['Geography', 'date_'])

totals = test.groupby(['date_', 'Geography'])

,date_,Geography,Variable,Value,Percentage
0,2000-01-01,"Austin-Round Rock-Georgetown, TX Metro Area",Government,138700.0,21.098266
1,2000-01-01,"Austin-Round Rock-Georgetown, TX Metro Area",Total Private,518700.0,78.901734
46,2000-02-01,"Austin-Round Rock-Georgetown, TX Metro Area",Government,142400.0,21.349325
47,2000-02-01,"Austin-Round Rock-Georgetown, TX Metro Area",Total Private,524600.0,78.650675
92,2000-03-01,"Austin-Round Rock-Georgetown, TX Metro Area",Government,142700.0,21.162687
...,...,...,...,...,...
13385,2024-03-01,"Yuba City, CA Metro Area",Total Private,37300.0,72.286822
13430,2024-04-01,"Yuba City, CA Metro Area",Government,14600.0,28.023033
13431,2024-04-01,"Yuba City, CA Metro Area",Total Private,37500.0,71.976967
13476,2024-05-01,"Yuba City, CA Metro Area",Government,14800.0,28.136882


In [36]:
totals = jobs_3.groupby(['date_', 'Geography']).sum().reset_index()

totals = totals.sort_values(by = ['Geography', 'date_'])

totals['Variable'] = 'Total'

totals = totals[['date_', 
                 'Geography',
                 'Variable',
                 'Value']]

display(totals.head(5))

,date_,Geography,Variable,Value
0,2000-01-01,National,Total,257986000
1,2000-02-01,National,Total,259310000
2,2000-03-01,National,Total,261504000
3,2000-04-01,National,Total,263498000
4,2000-05-01,National,Total,265412000


In [31]:
totals.to_excel("Jobs_1_MSA_BLS_SMU_Total.xlsx", index=False)

In [33]:
totals.to_csv("Jobs_1_MSA_BLS_SMU_Total.csv", index=False)

In [ ]:
with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), engine='xlsxwriter') as writer:
    # with pd.ExcelWriter(os.path.join(path_out_xlsx, name_output_xlsx), mode='a', engine='openpyxl', if_sheet_exists='replace') as writer:
    df_bls1_total.to_excel(writer, index = False, sheet_name = 'National'       )

In [ ]:
# Set output name for .xlsx files
name_output_xlsx = [indicator_name, ' ', geography, ' ', estimate, ' ', survey, '.xlsx']
name_output_xlsx = "".join(name_output_xlsx)

# Set output name for .csv files
name_output_csv = [indicator_name, '_', geography, '_', estimate, '_', survey, '.csv']
name_output_csv = "".join(name_output_csv)

print(name_output_xlsx)
print(name_output_csv )

# Set file path for exporting
path_out_xlsx = os.path.join(path_main, report_theme, sp_folder_out)
path_out_csv  = os.path.join(path_agol, indicator_name)

print('Excel files exported here: ' + path_out_xlsx)
print('CSV files exported here: '   + path_out_csv )


# Export to csv
df_bls1.to_csv(os.path.join(path_out_csv, name_output_csv), index = False)

In [11]:
path_config0

'C:\\Users\\jchoy\\Documents\\Python Projects\\Regional-Monitoring\\Indicator_Gen\\config'

In [13]:
pd.read_sas("C:\\Users\\jchoy\\Documents\\chis01_adult_sas_0.zip\\Data\\ADULT.xpt")

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\jchoy\\Documents\\chis01_adult_sas_0.zip\\Data\\ADULT.xpt'

***

## **Process Overview**

***

In using the Bureau of Labor Statistics (**BLS**) database for sourcing both the number of jobs (**Jobs_1, Jobs_2, Jobs_3**) and unemployment rates (**Labor_2**), we follow a simple process.

**Sourcing -**
$\newline$
BLS data can be viewed as tables manually on the web using their [data screen search tool](https://www.bls.gov/data/#employment). These tables are what we want to pull. In order to do so, we use BLS API Version 2. Version 2 allows users to query up to 500 times per day, run 50 series at a time, and pull 20 years per series. To use V2, the user must [register](https://data.bls.gov/registrationEngine/) by providing email, completing the captcha, and verifying their account from an email sent by BLS. In the email, the user will receive a key allowing them to use V2. 

**Series ID Creation -**
$\newline$
To use the API, we first need to understand how a series call works. The API call follows a basic structure: Prefix/Survey + Adjustment Code + Geography + Sector + Datatype. The prefix defines what survey you're using. For instance, if we want total jobs we use SM, but if we want unemployment rate we use LA. The adjustment code will determine if BLS provides the user with seasonally adjusted or unadjusted data. A user might want to be wary of this if they are concerned with temporary holiday workers for example. Depending on survey, geography can include many municipalities or only exist on certain levels. Recall the SM survey, in this survey we are able to pull on the MSA, County, and State level. However, we are not able to pull on a national level without changing the Prefix to CE. 

The sector correlates to the industry that a job applies to. In the SM survey, we have both supersector and industry code to consider. Supersectors are umbrellas for industries, and contain all the jobs that are considered to be this genre of career. For instance, a sector of 70000000 passed through the API call would correlate to all jobs in the Leisure-Hospitality industry, whilst 70712000 is considered part of the Leisure-Hospitality industry, but specifically correlates to jobs involving Museums, Historical Sites, and Similar Institutions. 

Finally, datatype refers to the format in which the data is returned. This could be numerical total, thousands, percent, etc. In the code, the function works by taking the configuration inputs defined by the user in the Excel file "BLS Configuration File". To use this file, the user defines the indicator_name, sample, geography, percentages, number of variables, and the years they want to pull. For the indicator_name they choose, the user must go into the config file and select the specific sectors or industries they want to include in the query. Depending on the geography level the user defines, it will determine whether MSAs or National totals are used. 

**Function Process -**
$\newline$
From here, the inputs are read into the local environment and used to run queries. The function `dict_maker`creates each Series ID for the respective indicator and geography. This is stored in a dictionary with keys being the Series ID and the Values being the geography. From here, we pass the data along to the actual call function, `bls_query_update`. `bls_query_update` takes these Series IDs, and performs the API call for each one, storing the associated data into a column for each geography. With our final function `full_bls`, we incorporate both `dict_maker` and `bls_query_update`into one function, that creates multiple dataframes for each specific sector we provide. From here, we clean the data to prepare for export. 

**Exporting/Cleaning -**
$\newline$
To clean the data, we rename the sectors to the SACOG definitions as given by Garett. If percentages is set to yes, we calculate the percentages by grouping together the unique year and MSA. Within the group, we calculate the percent makeup for each industry and add this as a new column. We then need to append all the data frames together and include the variable name as a column in our final frame. In the end, we should have four columns: date, Geography, Total, Industry. Once done, we export the data directly to SharePoint using the user_path at the beginning of the script. We also export to the local directory of the script's location. 

These can be referenced in the [series ID help page](https://www.bls.gov/help/hlpforma.htm#EN).

**Considerations -**
$\newline$
When querying BLS data, it's important to understand how many series calls one has per day. Version 2 users have 500 queries per day, with 50 series per query. Additionally, per query you can pull up to 20 years worth of data and anything more than that becomes a new query. Because of this, the code had to be changed to be able to handle running two queries for sourcing data in the range of 2000-2024.

Currently, the code is limited to handling only three specific surveys: SM (State and Area Employment, Hours, and Earnings), CE (National Employment, Hours, and Earnings), and LA (Local Area Unemployment). As a result, users are unable to make custom calls with other surveys. Moreover, the current code must be adapted for every new survey introduced.

Another issue that might arise when querying BLS, is the case when data does not exist for a series at all. In this case, you can paste the API call [URL](https://api.bls.gov/publicAPI/v2/timeseries/data/), followed by the series ID you want to test into your browser. If there is no data available, then the series simply does not exist. For example, let's try and find the unemployment rate of the Sacramento county. The associated Series ID would be LAUCA0647200000000003. If we put this [URL](https://api.bls.gov/publicAPI/v2/timeseries/data/LAUCA0647200000000003) into the browser, we see that it returns that this series does not exist. However, if we do this for the entire state of California (LAUST060000000000003), we see that a series is returned with values. Additionally, if you pull an empty series, this still counts toward the query limit. 

It's also important to note that the HTML output does not show all of the years of available data for a given series. The user should be able to pull data that goes well beyond what is given by the output, it just depends on survey. 